# PaddleOCR Testing

**WeatherSpeak PH** — Gemma 4 Hackathon

## About PaddleOCR

[PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR) is a production-grade OCR toolkit that:
- Supports **100+ languages**
- **75k+ stars**, most popular OCR library
- Actively maintained (updated 3 days ago)
- Industrial-strength, used in production worldwide
- PP-Structure for layout analysis and table recognition
- Built for RAG and document understanding workflows

### Why Test PaddleOCR?
PaddleOCR is the **most mature traditional deep learning OCR**. It represents the battle-tested baseline that production systems rely on.

## 1. Install PaddleOCR

In [ ]:
# Install PaddlePaddle (CPU version for compatibility)
!pip install -q paddlepaddle==2.6.1

# Install PaddleOCR
!pip install -q paddleocr

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image
import time
import numpy as np

from paddleocr import PaddleOCR

print("✓ PaddleOCR imported successfully")

## 2. Load Sample Data

Load the same sample images used for Surya OCR.

In [ ]:
# Load metadata
data_dir = Path("../data")
metadata_path = data_dir / "sample_metadata.json"

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

samples = metadata['samples']
print(f"Loaded {len(samples)} sample images")

# Create output directory for results
output_dir = data_dir / "paddleocr_results"
output_dir.mkdir(exist_ok=True)

## 3. Initialize PaddleOCR

Configure PaddleOCR for English text recognition.

In [ ]:
%%time
# Initialize PaddleOCR with English language
print("Loading PaddleOCR models (this may download models on first run)...")

ocr = PaddleOCR(
    lang='en',
    use_angle_cls=True,  # Enable text angle classification
    show_log=False       # Reduce verbose logging
)

print("✓ PaddleOCR initialized")

## 4. Run OCR on Sample Images

Process each sample image and extract text.

In [ ]:
results = []

for i, sample in enumerate(samples, 1):
    print(f"\n{'='*60}")
    print(f"Processing {i}/{len(samples)}: {sample['filename']}")
    print(f"{'='*60}")
    
    # Run OCR
    start_time = time.time()
    result = ocr.ocr(sample['image_path'], cls=True)
    processing_time = time.time() - start_time
    
    # Extract text from predictions
    # PaddleOCR returns [[bbox, (text, confidence)], ...]
    text_lines = []
    if result and result[0]:
        for line in result[0]:
            bbox = line[0]
            text = line[1][0]
            confidence = line[1][1]
            text_lines.append({
                'text': text,
                'confidence': confidence,
                'bbox': bbox
            })
    
    full_text = '\n'.join([line['text'] for line in text_lines])
    
    # Store results
    result_data = {
        'filename': sample['filename'],
        'storm': sample['storm'],
        'processing_time': processing_time,
        'num_lines': len(text_lines),
        'avg_confidence': sum([line['confidence'] for line in text_lines]) / len(text_lines) if text_lines else 0,
        'full_text': full_text,
        'text_lines': text_lines
    }
    results.append(result_data)
    
    # Display summary
    print(f"\n📊 Results:")
    print(f"  Processing time: {processing_time:.2f}s")
    print(f"  Lines detected: {len(text_lines)}")
    print(f"  Average confidence: {result_data['avg_confidence']:.2%}")
    print(f"  Characters extracted: {len(full_text)}")
    print(f"\n📄 Extracted text (first 500 characters):")
    print("-" * 60)
    print(full_text[:500])
    print("-" * 60)

## 5. Save Results

Save OCR results for comparison in the analysis notebook.

In [ ]:
# Save results to JSON
results_path = output_dir / "paddleocr_results.json"
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to: {results_path}")

# Also save full text files for easy reading
for result in results:
    text_path = output_dir / f"{Path(result['filename']).stem}_text.txt"
    with open(text_path, 'w', encoding='utf-8') as f:
        f.write(result['full_text'])

print(f"✓ Text files saved to: {output_dir}")

## 6. Performance Summary

Aggregate performance metrics across all samples.

In [ ]:
import statistics

processing_times = [r['processing_time'] for r in results]
line_counts = [r['num_lines'] for r in results]
avg_confidences = [r['avg_confidence'] for r in results if r['avg_confidence'] > 0]

print("\n" + "="*60)
print("PADDLEOCR PERFORMANCE SUMMARY")
print("="*60)
print(f"\n📊 Processing Time:")
print(f"  Average: {statistics.mean(processing_times):.2f}s")
print(f"  Min: {min(processing_times):.2f}s")
print(f"  Max: {max(processing_times):.2f}s")
print(f"\n🎯 Confidence:")
print(f"  Average: {statistics.mean(avg_confidences):.2%}")
print(f"  Min: {min(avg_confidences):.2%}")
print(f"  Max: {max(avg_confidences):.2%}")
print(f"\n📄 Content:")
print(f"  Average lines per page: {statistics.mean(line_counts):.0f}")
print(f"  Total samples processed: {len(results)}")
print(f"\n✓ All results saved to: {output_dir.absolute()}")

## 7. Visual Inspection

Display one sample with its extracted text for manual quality assessment.

In [ ]:
# Display first sample
if results:
    sample_result = results[0]
    sample_info = samples[0]
    
    print(f"Sample: {sample_result['filename']}")
    print(f"Storm: {sample_result['storm']}")
    print(f"Average Confidence: {sample_result['avg_confidence']:.2%}")
    print(f"\n" + "="*60)
    
    # Show image
    img = Image.open(sample_info['image_path'])
    display(img.resize((600, int(600 * img.size[1] / img.size[0]))))
    
    # Show extracted text
    print("\n📄 EXTRACTED TEXT:")
    print("="*60)
    print(sample_result['full_text'])
    print("="*60)

## 8. Preliminary Assessment

### ✅ Strengths
- Industry-proven reliability
- Confidence scores for quality assessment
- Extensive language support
- Large community and ecosystem
- Production-ready deployment

### ❓ Questions for Comparison
- How does accuracy compare to Surya OCR?
- Is processing speed faster or slower?
- Can Gemma 4 Vision match this reliability?
- How well does it preserve document structure?

### 📝 Next Steps
Proceed to **Notebook 04** to test Gemma 4 Vision - the key hypothesis for this hackathon project.